# Quality Assessment Product Price vs Selling Price Analysis


Import libraries

In [ ]:
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Project Eniac/Data/Data_clean/"

orders_cl = pd.read_csv(path + "orders_clean.csv")
orderlines = pd.read_csv(path + "orderlines_qu.csv")
products = pd.read_csv(path + "products_clean.csv")


Mounted at /content/drive


## 1.&nbsp; Define Pandas display format

In [ ]:
pd.set_option("display.max_columns", None)

## 2.&nbsp; Merge orderlines with products

In [ ]:
orderlines_discount_analysis = orderlines.merge(
    products[["sku", "price"]],
    on="sku",
    how="left"
)

In [ ]:
orderlines_discount_analysis.head()

,id,id_order,product_id,product_quantity,sku,unit_price,date,unit_price_total,price
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19,18.99,34.99
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45,399.00,429.00
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57,474.05,699.00
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40,68.39,79.00
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38,23.74,29.90


The merged dataset now contains both the catalog price and the actual selling price for each sold product.

It is now ready for discount analysis.

## 3.&nbsp; Calculate the discount

Calculate both the absolute discount amount and the percentage discount.

In [ ]:
orderlines_discount_analysis["discount_amount"] = (
    orderlines_discount_analysis["price"] -
    orderlines_discount_analysis["unit_price"]
)

In [ ]:
orderlines_discount_analysis["discount_pct"] = (
    orderlines_discount_analysis["discount_amount"]
    / orderlines_discount_analysis["price"] * 100
).round(2)

In [ ]:
orderlines_discount_analysis["discount_pct"].describe()

,discount_pct
count,212546.000000
mean,19.925325
std,24.792647
min,-4905.010000
25%,6.750000
50%,16.030000
75%,26.690000
max,200.000000


Identify anomalies

In [ ]:
anomalies = orderlines_discount_analysis[
    orderlines_discount_analysis["discount_pct"] < 0
]

anomalies.head()

,id,id_order,product_id,product_quantity,sku,unit_price,date,unit_price_total,price,discount_amount,discount_pct
27,1119162,299567,0,1,APP1647,769.0,2017-01-01 03:16:51,769.0,639.0,-130.0,-20.34
40,1119196,299583,0,1,APP1648,879.0,2017-01-01 08:54:20,879.0,749.0,-130.0,-17.36
52,1119211,299593,0,1,APP1651,879.0,2017-01-01 09:40:11,879.0,749.0,-130.0,-17.36
57,1119218,296284,0,1,BNQ0042,699.0,2017-01-01 09:58:35,699.0,694.0,-5.0,-0.72
81,1119277,299618,0,1,APP1651,879.0,2017-01-01 11:13:03,879.0,749.0,-130.0,-17.36


In [ ]:
len(anomalies)

8427

Inspect extreme discounts

In [ ]:
valid_discounts = orderlines_discount_analysis[
    orderlines_discount_analysis["unit_price"] >= 0
]

valid_discounts.sort_values(
    "discount_pct",
    ascending=False
)[["sku", "price", "unit_price", "discount_pct"]].head(10)

,sku,price,unit_price,discount_pct
167735,OWC0214,362.99,0.0,100.0
153184,MIC0035,265.99,0.0,100.0
192005,NTE0013,26.99,0.0,100.0
159996,SPE0189,49.99,0.0,100.0
159995,BEL0298,24.99,0.0,100.0
192172,NTE0013,26.99,0.0,100.0
208188,CRU0052-2,217.98,0.0,100.0
192201,NTE0013,26.99,0.0,100.0
151804,IFX0056,26.99,0.0,100.0
153196,MIC0035,265.99,0.0,100.0


### Conclusion
The comparison between orderlines.unit_price and products.price confirms that discounts are commonly applied, with an average discount of approximately 20%. Most products are sold below their catalog price, while a smaller proportion are sold at the original price without discounts.
During the analysis, 8,427 order lines (approximately 4% of the dataset) were identified as anomalies. These records include negative discount percentages (where the selling price exceeds the catalog price), free products (unit_price = 0), and negative selling prices. These cases are unlikely to represent standard discount transactions and may instead reflect pricing updates, promotional giveaways, product replacements, refunds, or data quality issues.
Since these records do not represent regular discount behavior, they were excluded from the discount analysis but retained in a separate dataset for further investigation.

In [ ]:
clean_discount_analysis = orderlines_discount_analysis[
    (orderlines_discount_analysis["unit_price"] > 0) &
    (orderlines_discount_analysis["discount_pct"] >= 0) &
    (orderlines_discount_analysis["discount_pct"] <= 100)
].copy()

In [ ]:
discount_anomalies = orderlines_discount_analysis[
    (orderlines_discount_analysis["unit_price"] <= 0) |
    (orderlines_discount_analysis["discount_pct"] < 0) |
    (orderlines_discount_analysis["discount_pct"] > 100)
].copy()

In [ ]:
path = "/content/drive/MyDrive/Project Eniac/Data/Data_clean/"


clean_discount_analysis.to_csv(path + "discount_analysis_clean.csv", index=False)
discount_anomalies.to_csv(path + "discount_anomalies.csv", index=False)

In [20]:
clean_discount_analysis.head()

,id,id_order,product_id,product_quantity,sku,unit_price,date,unit_price_total,price,discount_amount,discount_pct
0,1119109,299539,0,1,OTT0133,18.99,2017-01-01 00:07:19,18.99,34.99,16.00,45.73
1,1119110,299540,0,1,LGE0043,399.00,2017-01-01 00:19:45,399.00,429.00,30.00,6.99
2,1119111,299541,0,1,PAR0071,474.05,2017-01-01 00:20:57,474.05,699.00,224.95,32.18
3,1119112,299542,0,1,WDT0315,68.39,2017-01-01 00:51:40,68.39,79.00,10.61,13.43
4,1119113,299543,0,1,JBL0104,23.74,2017-01-01 01:06:38,23.74,29.90,6.16,20.60
